In [ ]:
import re
from pathlib import Path
import pathlib as pl

import numpy as np
import geopandas as gpd
import xarray as xr

from pyproj import Geod


## New Files Name


In [ ]:

# === USER INPUTS ===
input_folder = pl.Path(r'path\to\your\folder')  # Folder with ICESat-2 .h5 files
coastline_path = pl.Path(r'path\to\your\shoreline.shp')  # Coastline shapefile
output_folder = input_folder / "filtered"  # Output folder
beam_groups = ['gt1l', 'gt1r', 'gt2l', 'gt2r', 'gt3l', 'gt3r']
buffer_dist = 600  # meters

WRITE_GPKG = False         # set True if you want GeoPackage too
WRITE_GEOPARQUET = False    # modern + fast format

# === HELPERS ===
GEOD = Geod(ellps="WGS84")

def cumdist_geodesic(lon, lat):
    """
    Cumulative geodesic distance (meters) along the given lon/lat sequence.
    Assumes the sequence is already ordered the way you want (e.g., N->S).
    """
    lon = np.asarray(lon, dtype=float)
    lat = np.asarray(lat, dtype=float)
    n = len(lon)
    if n == 0:
        return np.array([], dtype=float)
    if n == 1:
        return np.array([0.0], dtype=float)
    # pairwise distances
    _, _, d = GEOD.inv(lon[:-1], lat[:-1], lon[1:], lat[1:])
    return np.concatenate(([0.0], np.cumsum(d)))

def safe_first(arr):
    try:
        return arr[0]
    except Exception:
        return None

def parse_date_track_from_name(h5_path):
    """
    Parse YYYYMMDD and 4-digit track_id from ATL06 filenames like:
    ATL06_20190105212430_01290203_006_01.h5
                ^^^^^^^^  ^^^^
    Returns (date_str, track_id) or (None, None) if not found.
    """
    stem = Path(h5_path).stem
    parts = stem.split('_')

    date = None
    track_id = None

    # Primary: strict per your rule
    if len(parts) >= 3:
        # parts[1] = 'YYYYMMDDHHMMSS' → take first 8
        if parts[1].isdigit() and len(parts[1]) >= 8:
            date = parts[1][:8]
        # parts[2] = '01290203' → take first 4 as track
        if parts[2].isdigit() and len(parts[2]) >= 4:
            track_id = parts[2][:4]

    # Fallback: regex (handles minor naming variations)
    if date is None or track_id is None:
        m = re.search(r'_(\d{8})(?:\d{6})?_([0-9]{4})', stem)
        if m:
            date = date or m.group(1)
            track_id = track_id or m.group(2)

    return date, track_id

# === PREP ===
output_folder.mkdir(exist_ok=True)

# Read coastline and build a single buffer polygon in meters CRS (EPSG:3413)
coastline = gpd.read_file(coastline_path).to_crs("EPSG:3413")
# If the shoreline layer has multiple parts, .buffer() then unary_union gives one geometry
coast_buffer_geom = coastline.buffer(buffer_dist).union_all()  # shapely (Multi)Polygon

# === PROCESS EACH .H5 FILE ===
for h5_file in sorted(input_folder.glob("*.h5")):
    print(f"\n📂 Processing: {h5_file.name}")
    try:
        for beam in beam_groups:
            try:
                group = f'/{beam}/land_ice_segments'
                # Open as context to ensure clean close
                with xr.open_dataset(h5_file, group=group, engine='h5netcdf') as ds:
                    # Required vars (drop NaNs right away)
                    if not all(v in ds.variables for v in ['latitude', 'longitude', 'h_li']):
                        print(f"  ⚠️  {beam}: missing required vars; skipping.")
                        continue

                    lat = ds['latitude'].values
                    lon = ds['longitude'].values
                    h_li = ds['h_li'].values

                    # Quality filter (optional; comment out if you don't want it)
                    # Keep 'good' segments only if available
                    if 'atl06_quality_summary' in ds.variables:
                        q = ds['atl06_quality_summary'].values
                        good = (q == 0)
                        lat, lon, h_li = lat[good], lon[good], h_li[good]

                    # Drop NaNs
                    m = np.isfinite(lat) & np.isfinite(lon) & np.isfinite(h_li)
                    lat, lon, h_li = lat[m], lon[m], h_li[m]
                    if lat.size == 0:
                        print(f"  ⚠️  {beam}: no valid points after QC/NaN filter.")
                        continue

                    # Metadata: date & track (prefer variables, fallback to filename)
                    date_str, track_id_name = parse_date_track_from_name(h5_file)

                    # RGT and cycle (if present)
                    rgt = None
                    cycle = None
                    if 'rgt' in ds.variables:
                        try:
                            rgt = int(np.nanmedian(ds['rgt'].values))
                        except Exception:
                            pass
                    if 'cycle_number' in ds.variables:
                        try:
                            cycle = int(np.nanmedian(ds['cycle_number'].values))
                        except Exception:
                            pass
                    # Fallbacks
                    if track_id_name is None and rgt is not None:
                        track_id_name = f"{rgt:04d}"

                    # Build GeoDataFrame in lon/lat, then project to meters CRS for clipping
                    gdf = gpd.GeoDataFrame(
                        {
                            'latitude': lat,
                            'longitude': lon,
                            'h_li': h_li,
                            'track_id': track_id_name,
                            'gt': beam,
                            'date': date_str,
                            'rgt': rgt,
                            'cycle': cycle,
                        },
                        geometry=gpd.points_from_xy(lon, lat),
                        crs="EPSG:4326",
                    ).to_crs("EPSG:3413")

                    # --- EARLY CLIP TO BUFFER (includes boundary) ---
                    # For points, 'intersects' behaves like "inside or on boundary"
                    in_buf = gdf.geometry.intersects(coast_buffer_geom)
                    selected = gdf.loc[in_buf].copy()

                    if selected.empty:
                        print(f"  ⚠️  {beam}: no points within {buffer_dist} m buffer.")
                        continue

                    # Restore lon/lat columns after projection (keep both CRSes if you like)
                    selected_ll = selected.to_crs("EPSG:4326")
                    selected['latitude'] = selected_ll.geometry.y.values
                    selected['longitude'] = selected_ll.geometry.x.values

                    # --- ORDER & DISTANCE: start at northernmost point ---
                    selected.sort_values('latitude', ascending=False, inplace=True)
                    # Use geodesic cumulative distance along lon/lat (meters)
                    dists = cumdist_geodesic(selected['longitude'].values,
                                             selected['latitude'].values)
                    selected['distance_m'] = dists

                    # --- WRITE OUTPUTS ---
                    out_stem = f"ATL06_{selected['track_id'].iloc[0] or 'unk'}_{beam}_{date_str or 'nodate'}"
                    shp_path = output_folder / f"{out_stem}.shp"
                    selected.to_file(shp_path)
                    print(f"  ✅ {beam}: {len(selected)} pts → {shp_path.name}")

                    if WRITE_GPKG:
                        gpkg_path = output_folder / f"{out_stem}.gpkg"
                        selected.to_file(gpkg_path, driver="GPKG")
                    if WRITE_GEOPARQUET:
                        parquet_path = output_folder / f"{out_stem}.parquet"
                        selected.to_parquet(parquet_path, index=False)

            except Exception as beam_error:
                print(f"  ⚠️  Skipping {beam} in {h5_file.name}: {beam_error}")

    except Exception as file_error:
        print(f"❌ Failed to process {h5_file.name}: {file_error}")
